# 🏥 Pre-Consultation Agent - NEW SYSTEM Testing (Kaggle)

This notebook tests the **complete new two-stage extraction system** with intelligent routing on Kaggle's FREE Tesla P100 GPU.

## What This Tests:
- ✅ **Model A**: Audio transcription + quality assessment
- ✅ **Model B Light**: Quick extraction for routing
- ✅ **Conversation Router**: Emergency/Rule-based/AI-powered paths
- ✅ **Model C Rules**: Predefined questions for known symptoms
- ✅ **Model C AI**: Adaptive questions for complex cases
- ✅ **Patient Info Collection**: Name, age, gender
- ✅ **Model B Full**: Comprehensive extraction after conversation
- ✅ **Models D-F**: Risk scoring, patient guidance, doctor summary
- ✅ **Cost Tracking**: API calls and cost estimation

## Setup Checklist:
1. ✅ Turn on GPU: **Settings → Accelerator → GPU P100**
2. ✅ Enable Internet: **Settings → Internet → ON**
3. ✅ Add Secrets: **Add-ons → Secrets**
   - `GEMINI_API_KEY` — for Models B, C, D, E, F (get from [Google AI Studio](https://aistudio.google.com/apikey))
   - `HF_TOKEN` — for Model A / Whisper download (get from [Hugging Face](https://huggingface.co/settings/tokens))
   - `NGROK_AUTH_TOKEN` — for exposing API to Flutter (get from [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken))
4. ✅ Upload Audio: **+ Add Data → Upload your WAV file**

**GPU Quota**: 30 hours/week

Let's test the new system! 🚀

## Step 1: Verify GPU and Environment

In [1]:
import torch
import os
import sys

print("🔍 Checking environment...\n")

# Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✅ GPU Available: {gpu_name}")
    print(f"   Memory: {gpu_memory:.1f}GB")
else:
    print("❌ No GPU detected!")
    print("   Go to Settings → Accelerator → GPU P100")

print(f"\n🐍 Python: {sys.version.split()[0]}")
print(f"📁 Working dir: {os.getcwd()}")

🔍 Checking environment...

❌ No GPU detected!
   Go to Settings → Accelerator → GPU P100

🐍 Python: 3.12.12
📁 Working dir: /kaggle/working


## Step 2: Clone/Update Repository

In [2]:
import os
from pathlib import Path

repo_path = Path('/kaggle/working/pre-consultation-agent')

if repo_path.exists():
    print("🔄 Repository exists - pulling latest changes...\n")
    os.chdir(str(repo_path))
    !git pull origin main
    print("\n✅ Latest code pulled!")
else:
    print("📥 Cloning repository...\n")
    !git clone https://github.com/buwituze/pre-consultation-agent.git /kaggle/working/pre-consultation-agent
    print("\n✅ Repository cloned!")

os.chdir('/kaggle/working/pre-consultation-agent/backend')
print(f"\n📁 Current directory: {os.getcwd()}")

📥 Cloning repository...

Cloning into '/kaggle/working/pre-consultation-agent'...
fatal: unable to access 'https://github.com/buwituze/pre-consultation-agent.git/': Could not resolve host: github.com

✅ Repository cloned!


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/pre-consultation-agent/backend'

## Step 3: Install Dependencies

In [ ]:
print("📦 Installing dependencies...\n")
!pip install -q -r requirements.txt

print("✅ Dependencies installed!\n")

# Verify NEW system modules
import transformers
import google.genai
print(f"📚 Key versions:")
print(f"   transformers: {transformers.__version__}")
print(f"   google.genai: {google.genai.__version__}")

## Step 4: Configure API Keys

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# ---------- API keys & device ----------
try:
    user_secrets = UserSecretsClient()
    gemini_key = user_secrets.get_secret("GEMINI_API_KEY")
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("✅ Using Kaggle Secrets")
except Exception:
    print("⚠️ Kaggle Secrets not found - using manual setup")
    gemini_key = ''
    hf_token = ''

os.environ['GEMINI_API_KEY'] = gemini_key
os.environ['HF_TOKEN'] = hf_token
os.environ['DEVICE'] = 'cuda'
os.environ['MAX_TURNS'] = '10'
os.environ['SKIP_LANG_DETECTION_WHEN_HINTED'] = 'true'

print(f"\n✅ Environment configured:")
print(f"   GEMINI_API_KEY: {'✓ SET' if gemini_key and 'YOUR_' not in gemini_key else '❌ NOT SET'}")
print(f"   HF_TOKEN:       {'✓ SET' if hf_token and 'YOUR_' not in hf_token else '❌ NOT SET'}")
print(f"   DEVICE: cuda (GPU)")

if 'YOUR_' in gemini_key or 'YOUR_' in hf_token:
    print("\n⚠️ WARNING: Update missing API keys before continuing!")
    print("   Add them in: Add-ons → Secrets")

# ---------- Database configuration ----------
try:
    # assume same UserSecretsClient exists
    db_host = user_secrets.get_secret("DB_HOST")
    db_port = user_secrets.get_secret("DB_PORT")
    db_name = user_secrets.get_secret("DB_NAME")
    db_user = user_secrets.get_secret("DB_USER")
    db_pass = user_secrets.get_secret("DB_PASSWORD")
    print("✅ Loaded database credentials from Kaggle Secrets")
except Exception:
    print("⚠️ Database secrets not found. You can set them manually below.")
    db_host = ""  # e.g. xxxxxx.render.com or ngrok tcp host
    db_port = ""
    db_name = ""
    db_user = ""
    db_pass = ""

# export to environment for FastAPI server
os.environ['DB_HOST'] = db_host
os.environ['DB_PORT'] = db_port
os.environ['DB_NAME'] = db_name
os.environ['DB_USER'] = db_user
os.environ['DB_PASSWORD'] = db_pass
os.environ['USE_DB'] = 'true'

print(f"\n🔧 Database environment configured:")
print(f"   HOST={db_host}:{db_port}, DB={db_name}, USER={db_user}")

# optional quick test
try:
    import psycopg2
    conn = psycopg2.connect(
        host=db_host,
        port=db_port,
        database=db_name,
        user=db_user,
        password=db_pass,
        connect_timeout=5,
    )
    conn.close()
    print("✅ Successfully connected to database")
except Exception as e:
    print(f"❌ Database connection failed: {e}")

# ---------- Email configuration (EmailJS) ----------
try:
    emailjs_service_id  = user_secrets.get_secret("EMAILJS_SERVICE_ID")
    emailjs_template_id = user_secrets.get_secret("EMAILJS_TEMPLATE_ID")
    emailjs_public_key  = user_secrets.get_secret("EMAILJS_PUBLIC_KEY")
    emailjs_private_key = user_secrets.get_secret("EMAILJS_PRIVATE_KEY")
    print("✅ EmailJS credentials loaded from Kaggle Secrets")
except Exception:
    emailjs_service_id  = ""
    emailjs_template_id = ""
    emailjs_public_key  = ""
    emailjs_private_key = ""
    print("⚠️ EmailJS secrets not found - add EMAILJS_SERVICE_ID, EMAILJS_TEMPLATE_ID, EMAILJS_PUBLIC_KEY, EMAILJS_PRIVATE_KEY in Add-ons → Secrets")

os.environ['EMAIL_PROVIDER']        = 'emailjs'
os.environ['EMAILJS_SERVICE_ID']    = emailjs_service_id
os.environ['EMAILJS_TEMPLATE_ID']   = emailjs_template_id
os.environ['EMAILJS_PUBLIC_KEY']    = emailjs_public_key
os.environ['EMAILJS_PRIVATE_KEY']   = emailjs_private_key
os.environ['SMTP_FROM_NAME']        = 'Pre-Consultation System'
os.environ['SMTP_FROM_EMAIL']       = 'noreply@pre-consultation.app'
# Update APP_LOGIN_URL to your frontend URL (or your ngrok frontend URL)
os.environ['APP_LOGIN_URL']         = os.environ.get('APP_LOGIN_URL', 'http://localhost:3000/login')

email_ok = all([emailjs_service_id, emailjs_template_id, emailjs_public_key, emailjs_private_key])
print(f"   EmailJS: {'✓ CONFIGURED' if email_ok else '❌ NOT CONFIGURED - credential emails will fail'}")


## Step 5: Load Whisper Models (Model A)

In [ ]:
import sys
import time
sys.path.insert(0, '/kaggle/working/pre-consultation-agent/backend')

from models import model_a

print("🔄 Loading Whisper models on GPU...")
print("   First run: 3-5 minutes (downloading ~6GB)\n")

start = time.time()
model_a.load_models()
elapsed = time.time() - start

print(f"\n✅ Models loaded in {elapsed:.1f}s")

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / 1e9
    print(f"📊 GPU Memory: {allocated:.2f}GB")

## Step 6: Full Conversation Test - NEW SYSTEM

This tests the complete patient journey in 3 phases:

**Phase 1 (auto):** Audio → Transcription → Light Extraction → Routing Decision
**Phase 2 (interactive):** Patient Info + Symptom Questions (you answer as the patient)
**Phase 3 (auto):** Full Extraction → Risk Scoring → Patient Guidance → Doctor Summary

---

**🎤 Upload your audio file first:**
- Click **+ Add Data** → Upload WAV file
- File will be at: `/kaggle/input/<dataset-name>/<filename>.wav`

In [ ]:
# ============================================================
# PHASE 1: Audio → Transcription → Light Extraction → Routing
# ============================================================
import time
import os
from models import model_a, model_b, model_c, model_d, model_e, model_f
from models.model_c_rules import get_symptom_questions, get_patient_info_questions
from routing.conversation_router import route_conversation

print("="*70)
print("🏥 PHASE 1: TRANSCRIPTION + ROUTING")
print("="*70)

# Find uploaded WAV file
wav_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.wav'):
            wav_files.append(os.path.join(root, file))

if not wav_files:
    print("\n❌ No audio file found!")
    print("📤 Upload a WAV file: + Add Data → Upload")
    raise SystemExit("Upload audio first")

audio_path = wav_files[0]
print(f"\n🎤 Audio: {os.path.basename(audio_path)} ({os.path.getsize(audio_path)/(1024*1024):.2f}MB)\n")

with open(audio_path, 'rb') as f:
    audio_bytes = f.read()

# --- Model A: Transcription ---
print("─"*70)
print("MODEL A — Transcription + Quality Assessment")
print("─"*70)

start = time.time()
transcription = model_a.transcribe(audio_bytes, language_hint='kinyarwanda')
elapsed = time.time() - start

transcript = transcription['full_text']
language = transcription['dominant_language']
confidence = transcription['mean_confidence']
quality = transcription.get('quality', 'medium')

print(f"✅ Complete in {elapsed:.1f}s")
print(f"   Language: {language} | Confidence: {confidence:.0%} | Quality: {quality}")
print(f"\n📝 Transcript:\n   {transcript}\n")

# --- Model B Light: Quick Extraction ---
print("─"*70)
print("MODEL B LIGHT — Extract Routing Info")
print("─"*70)

start = time.time()
light_extraction = model_b.extract_light(transcript)
elapsed = time.time() - start

chief_complaint = light_extraction.get('chief_complaint', 'unknown')
severity = light_extraction.get('severity_estimate', 0)
red_flags = light_extraction.get('red_flags_present', False)

print(f"✅ Complete in {elapsed:.1f}s")
print(f"   Chief Complaint: {chief_complaint}")
print(f"   Severity: {severity}/10")
print(f"   Red Flags: {'🚨 YES' if red_flags else '✅ No'}\n")

# --- Conversation Router ---
print("─"*70)
print("ROUTER — Decide Conversation Path")
print("─"*70)

routing = route_conversation(
    light_extraction=light_extraction,
    transcription_quality=quality,
    language=language
)

mode = routing['mode']
reasoning = routing['reasoning']

print(f"🎯 Mode: {mode.upper()}")
print(f"   Reasoning: {reasoning}")

# Check if preset questions exist for this symptom
lang_key = "kinyarwanda" if language == "kinyarwanda" else "english"
preset_questions = get_symptom_questions(chief_complaint, lang_key)

if mode == "emergency":
    print(f"\n🚨 EMERGENCY — Patient will be fast-tracked with minimal questions")
elif mode == "rule_based" and preset_questions:
    print(f"\n📋 RULE-BASED — {len(preset_questions)} preset questions for '{chief_complaint}'")
elif mode == "ai_powered":
    if preset_questions:
        print(f"\n🤖 AI-POWERED — but preset questions exist, will use those first")
    else:
        print(f"\n🤖 AI-POWERED — no preset questions, Model C will generate them")

print(f"\n{'='*70}")
print("✅ Phase 1 complete! Run the next cell for the interactive conversation.")
print(f"{'='*70}")

### Phase 2: Interactive Patient Conversation

This cell simulates the real conversation flow. **You type answers as the patient.**

On the actual system, the patient would speak and Model A would transcribe their answer.
Here we skip that step and you type directly — the conversation logic being tested is identical.

The system will:
1. Ask patient info questions (name, age, gender) — always first
2. Ask symptom questions based on the routing decision:
   - **Rule-based**: Uses preset questions for known symptoms
   - **AI-powered**: Model C generates adaptive questions
   - **Emergency**: Minimal questions, fast-track to scoring

In [ ]:
### ============================================================
# PHASE 2: Interactive Patient Conversation
# ============================================================
print("="*70)
print("🏥 PHASE 2: PATIENT CONVERSATION")
print("="*70)
print("Type your answers below as if you are the patient.\n")

conversation_turns = []

# --- Step 1: Patient Info (Always First) ---
print("─"*70)
print("PATIENT INFO — Always asked first")
print("─"*70)

info_questions = get_patient_info_questions(lang_key)

patient_name = ""
patient_age = None
patient_gender = ""

for q_dict in info_questions:
    print(f"\n🤖 System asks: {q_dict['question']}")
    answer = input("👤 Your answer: ").strip()
    
    if q_dict['id'] == 'patient_name':
        patient_name = answer
    elif q_dict['id'] == 'patient_age':
        try:
            patient_age = int(answer)
        except ValueError:
            patient_age = 30
            print(f"   (Could not parse age, using default: {patient_age})")
    elif q_dict['id'] == 'patient_gender':
        patient_gender = answer
    
    conversation_turns.append({"question": q_dict['question'], "answer": answer})

print(f"\n✅ Patient info collected: {patient_name}, age {patient_age}, {patient_gender}")

# --- Step 2: Symptom Questions (Based on Routing) ---
print(f"\n{'─'*70}")
print(f"SYMPTOM QUESTIONS — {mode.upper()} PATH")
print("─"*70)

symptom_turns = []

if mode == "emergency":
    # Emergency: 1 quick question then fast-track
    eq = "Can you describe your main concern right now?" if lang_key == "english" else "Ni iki gikomeye ubu?"
    print(f"\n🚨 EMERGENCY — one quick question before fast-tracking\n")
    print(f"🤖 System asks: {eq}")
    answer = input("👤 Your answer: ").strip()
    symptom_turns.append({"question": eq, "answer": answer})

elif mode == "rule_based" and preset_questions:
    # Rule-based: use preset questions for this symptom
    print(f"\n📋 Using {len(preset_questions)} preset questions for '{chief_complaint}'\n")
    for i, q_dict in enumerate(preset_questions, 1):
        print(f"\n🤖 Q{i}: {q_dict['question']}")
        answer = input("👤 Your answer: ").strip()
        symptom_turns.append({"question": q_dict['question'], "answer": answer})

else:
    # AI-powered: Model C generates adaptive questions
    max_questions = 4
    print(f"\n🤖 Model C will generate up to {max_questions} adaptive questions\n")
    
    for i in range(1, max_questions + 1):
        try:
            question = model_c.select_next_question(
                extraction=light_extraction,
                questions_asked=[t["question"] for t in symptom_turns],
                patient_answers=[t["answer"] for t in symptom_turns]
            )
            print(f"\n🤖 AI Q{i}: {question}")
            answer = input("👤 Your answer: ").strip()
            symptom_turns.append({"question": question, "answer": answer})
            
            # Check coverage after each answer
            combined_extraction = dict(light_extraction)
            if model_c.is_coverage_complete(combined_extraction, len(symptom_turns), int(os.environ.get('MAX_TURNS', 10))):
                print(f"\n✅ Coverage complete after {len(symptom_turns)} questions")
                break
        except Exception as e:
            print(f"\n⚠️ Model C error: {e}")
            break

conversation_turns.extend(symptom_turns)

print(f"\n{'='*70}")
print(f"✅ Conversation complete!")
print(f"   Patient info questions: {len(info_questions)}")
print(f"   Symptom questions: {len(symptom_turns)}")
print(f"   Total turns: {len(conversation_turns)}")
print(f"\n📝 Full conversation log:")
for i, turn in enumerate(conversation_turns, 1):
    print(f"   {i}. Q: {turn['question']}")
    print(f"      A: {turn['answer']}")
print(f"\n{'='*70}")
print("Run the next cell for extraction, scoring, and results.")
print(f"{'='*70}")

### Phase 3: Full Extraction + Scoring + Results

This runs the remaining models using the conversation data from Phase 2:
- **Model B Full**: Comprehensive extraction with all conversation context
- **Model D**: Risk scoring and priority assignment
- **Model E**: Patient guidance message
- **Model F**: Doctor clinical summary

In [ ]:
# ============================================================
# PHASE 3: Full Extraction + Scoring + Results
# ============================================================
import time

print("="*70)
print("🏥 PHASE 3: EXTRACTION + SCORING + RESULTS")
print("="*70)

# --- Model B Full: Comprehensive Extraction ---
print("\n" + "─"*70)
print("MODEL B FULL — Comprehensive Extraction (with conversation context)")
print("─"*70)

start = time.time()
try:
    full_extraction = model_b.extract_full(
        transcript=transcript,
        conversation_history=conversation_turns,
        target_language=language
    )
    elapsed = time.time() - start
    
    print(f"✅ Complete in {elapsed:.1f}s")
    print(f"\n📊 Extracted Clinical Data:")
    for key, value in full_extraction.items():
        if value and value != "unknown":
            print(f"   {key}: {value}")
except Exception as e:
    print(f"❌ Full extraction failed: {e}")
    full_extraction = light_extraction

# --- Model D: Risk Scoring ---
print("\n" + "─"*70)
print("MODEL D — Risk Scoring")
print("─"*70)

start = time.time()
try:
    risk_score = model_d.score(full_extraction, age=patient_age)
    elapsed = time.time() - start
    
    print(f"✅ Complete in {elapsed:.1f}s")
    print(f"\n🎯 Risk Assessment:")
    print(f"   Priority: {risk_score.get('priority', 'N/A')}")
    print(f"   Suspected Issue: {risk_score.get('suspected_issue', 'N/A')}")
    print(f"   Risk Factors: {risk_score.get('risk_factors', [])}")
    print(f"   Confidence: {risk_score.get('confidence', 'N/A')}")
except Exception as e:
    print(f"❌ Risk scoring failed: {e}")
    risk_score = {"priority": "MEDIUM", "suspected_issue": "general or systemic complaint", "risk_factors": [], "confidence": 0.5}

# --- Model E: Patient Guidance ---
print("\n" + "─"*70)
print("MODEL E — Patient Guidance Message")
print("─"*70)

start = time.time()
try:
    patient_guidance = model_e.generate_message(
        extraction=full_extraction,
        score=risk_score,
        language=language,
        location="Waiting Area A"
    )
    elapsed = time.time() - start
    
    print(f"✅ Complete in {elapsed:.1f}s")
    print(f"\n💬 Message to Patient:")
    print(f"   {patient_guidance}")
except Exception as e:
    print(f"❌ Patient guidance failed: {e}")
    patient_guidance = "Please wait to see the doctor."

# --- Model F: Doctor Summary ---
print("\n" + "─"*70)
print("MODEL F — Doctor Clinical Summary")
print("─"*70)

start = time.time()
try:
    doctor_summary = model_f.generate_brief(
        session_id="test-session-001",
        extraction=full_extraction,
        score=risk_score,
        questions_asked=[t["question"] for t in conversation_turns],
        patient_answers=[t["answer"] for t in conversation_turns],
        transcript=transcript,
        language=language,
        patient_age=patient_age
    )
    elapsed = time.time() - start
    
    print(f"✅ Complete in {elapsed:.1f}s")
    print(f"\n👨‍⚕️ Doctor Summary:")
    for key, value in doctor_summary.items():
        print(f"   {key}: {value}")
except Exception as e:
    print(f"❌ Doctor summary failed: {e}")
    doctor_summary = {}

# --- Final Summary ---
print("\n" + "="*70)
print("📊 SESSION COMPLETE")
print("="*70)

print(f"\n🎯 Routing: {mode.upper()} ({reasoning})")
print(f"👤 Patient: {patient_name}, age {patient_age}, {patient_gender}")
print(f"🩺 Complaint: {chief_complaint} (severity {severity}/10)")
print(f"⚡ Priority: {risk_score.get('priority', 'N/A')}")
print(f"📋 Total conversation turns: {len(conversation_turns)}")
print(f"\n{'='*70}")
print("✅ FULL PIPELINE TEST COMPLETE (Model A → F)")
print(f"{'='*70}")

## Summary of Pipeline Tests

✅ **What We Tested:**
- Model A: Transcription + quality assessment
- Model B: Light extraction → Full extraction
- Routing: All 3 paths (emergency/rule-based/ai-powered)
- Model C: Rule-based questions + AI questions
- Model D: Risk scoring
- Model E: Patient guidance
- Model F: Doctor summary
- Cost tracking & comparison

---

## Step 9: Deploy API with ngrok (For Frontend Testing)

This starts the FastAPI server and exposes it via ngrok so you can
connect your Flutter frontend.

### Setup:
1. Get a free ngrok auth token from [ngrok.com](https://dashboard.ngrok.com/get-started/your-authtoken)
2. Add it as a Kaggle Secret: **Add-ons → Secrets → Add `NGROK_AUTH_TOKEN`**
3. Run the cells below

### Usage:
- Copy the ngrok URL printed below
- Paste it as the API base URL in your Flutter app
- The server will stay running as long as the Kaggle session is active

In [ ]:
!pip install -q pyngrok

import os
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    ngrok_token = user_secrets.get_secret("NGROK_AUTH_TOKEN")
    print("✅ ngrok auth token loaded from Kaggle Secrets")
except:
    ngrok_token = ""
    print("⚠️ Set your ngrok token below or add it as a Kaggle Secret")

os.environ['NGROK_AUTH_TOKEN'] = ngrok_token

if 'YOUR_' in ngrok_token:
    print("\n❌ Replace YOUR_NGROK_AUTH_TOKEN_HERE with your actual token!")
    print("   Get one free at: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    print("✅ ngrok ready to launch")

### Step 9b: Start FastAPI Server + ngrok Tunnel

**⚠️ Important**: This cell will **block** — the server runs until you stop it.
The ngrok URL will be printed below. Copy it and use it in your Flutter app.

**Note on Database**: The kiosk endpoints (`/kiosk/start`, `/kiosk/finish`) require a database connection. 
If you don't have one configured, the server will still start but those endpoints will fail.
See the notes at the bottom of this notebook for database options.

In [ ]:
import subprocess, os

# Kill any existing uvicorn
subprocess.run("pkill -f uvicorn", shell=True)
subprocess.run("pkill -f 'main:app'", shell=True)

# Free GPU memory
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU free: {torch.cuda.mem_get_info()[0]/1e9:.1f}GB")

In [ ]:
import subprocess, os

# Install real ngrok binary via official apt repo (avoids pyngrok CDN which Kaggle blocks)
NGROK_BIN = '/tmp/ngrok_real'
if not os.path.exists(NGROK_BIN):
    print('Installing ngrok via apt...')
    subprocess.run('curl -sSL https://ngrok-agent.s3.amazonaws.com/ngrok.asc | tee /etc/apt/trusted.gpg.d/ngrok.asc > /dev/null', shell=True, check=True)
    subprocess.run('echo "deb https://ngrok-agent.s3.amazonaws.com buster main" | tee /etc/apt/sources.list.d/ngrok.list', shell=True, check=True)
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-q', 'ngrok'], check=True)
    # apt installs to /usr/bin/ngrok or /usr/local/bin/ngrok — copy to our path
    for candidate in ['/usr/bin/ngrok', '/usr/local/bin/ngrok']:
        if os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            # check it is the real binary, not pyngrok wrapper (real binary is >10MB)
            if os.path.getsize(candidate) > 5_000_000:
                import shutil
                shutil.copy2(candidate, NGROK_BIN)
                os.chmod(NGROK_BIN, 0o755)
                print(f'ngrok installed (copied from {candidate}).')
                break
    else:
        raise RuntimeError('ngrok binary not found after apt install')
else:
    print('ngrok already installed.')

# Point pyngrok to the real binary (not the pyngrok wrapper)
from pyngrok import conf, ngrok
conf.get_default().ngrok_path = NGROK_BIN

import os
import subprocess
import time
import requests
import threading

# helper to stream process output in background threads
def read_output(pipe, name):
    for line in iter(pipe.readline, b''):
        print(f"[{name}] {line.decode().rstrip()}")

# Authenticate ngrok
ngrok.set_auth_token(os.environ['NGROK_AUTH_TOKEN'])

# Change to backend directory
os.chdir('/kaggle/working/pre-consultation-agent/backend')

# NOTE: Do not force-disable the database here; if you have configured
# DB_HOST/DB_USER/etc above, the environment already contains USE_DB='true'.
# The old version of this cell always set USE_DB='false' which prevented
# kiosk endpoints from working when a remote database was available.
# Uncomment the following line only when you explicitly want to run without
# a database:
# os.environ['USE_DB'] = 'false'

# Start uvicorn as a subprocess (avoids asyncio.run() conflict in notebooks)
port = 8000
process = subprocess.Popen(
    ['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', str(port)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    env={**os.environ}
)

# start threads to capture output immediately
stdout_thread = threading.Thread(target=read_output, args=(process.stdout, "STDOUT"), daemon=True)
stderr_thread = threading.Thread(target=read_output, args=(process.stderr, "STDERR"), daemon=True)
stdout_thread.start()
stderr_thread.start()

# Wait for server to be ready
print("⏳ Starting FastAPI server...")
server_ready = False
for i in range(60):  # give twice as long for slow startup
    time.sleep(2)
    try:
        r = requests.get(f'http://localhost:{port}/health')
        if r.status_code == 200:
            server_ready = True
            break
    except requests.ConnectionError:
        pass

# Diagnostic report
if server_ready:
    print("✅ Server responded to health check")
else:
    print("❌ Server NOT responding after 120 seconds")
    print("\nServer output (last few lines):")
    try:
        stdout, stderr = process.communicate(timeout=5)
        print(stderr.decode())
    except Exception as e:
        print(f"(could not fetch output: {e})")
    raise RuntimeError("Server startup timeout")

print("✅ FastAPI server started!\n")

# Create ngrok tunnel
# Note: request_header_add is only supported in ngrok v3+; the Kaggle-bundled
# pyngrok uses ngrok v2 which doesn't support it. It's not needed anyway —
# ngrok's browser interstitial only affects browser navigation, not API requests.
print("🌐 Creating ngrok tunnel...")
try:
    ngrok_tunnel = ngrok.connect(port, "http")
    ngrok_url = ngrok_tunnel.public_url
    print(f"✅ ngrok tunnel created!")
    print(f"\n🚀 API URL: {ngrok_url}")
    print(f"\n📋 Use this in your Flutter app:")
    print(f"   BaseURL = '{ngrok_url}'")
    print(f"\n⏱️  Tunnel will stay active as long as this cell is running...")
    
    # Test the tunnel
    time.sleep(2)
    try:
        test = requests.get(f'{ngrok_url}/health', timeout=5)
        if test.status_code == 200:
            print(f"✅ Tunnel is working! API is accessible from outside Kaggle")
        else:
            print(f"⚠️ Tunnel created but health check returned {test.status_code}")
    except Exception as e:
        print(f"⚠️ Could not test tunnel: {e}")
        
except Exception as e:
    print(f"❌ ngrok tunnel failed: {e}")
    print(f"\nCommon issues:")
    print(f"1. Invalid NGROK_AUTH_TOKEN")
    print(f"2. ngrok account quota exceeded")
    print(f"3. Network connectivity issue")
    raise

# Keep the process running
print(f"\n{'='*70}")
print("⏳ Server and tunnel running... Press Ctrl+C or stop cell to exit")
print(f"{'='*70}")

try:
    process.wait()
except KeyboardInterrupt:
    print("\n\n🛑 Shutting down...")
    process.terminate()
    process.wait()
    print("✅ Shutdown complete")

## Database Notes & Connecting to Your Local DB

### Can the conversation system work without a database?

**The AI pipeline (Models A → F) works perfectly without a database.** The pipeline test cells above (Steps 1-8) prove this — they call the models directly with no DB involved.

However, the **kiosk API endpoints** that the Flutter frontend calls (`/kiosk/start`, `/kiosk/finish`) **DO require a database** because:
- `/kiosk/start` creates a patient record + session record in PostgreSQL
- `/kiosk/finish` persists audio references, conversation messages, symptoms, and predictions to the DB

The mid-conversation endpoints (`/kiosk/{id}/audio`, `/kiosk/{id}/answer`) use the **in-memory session store** and will work fine without a DB.

### How to connect Kaggle → your local PostgreSQL

**Option 1: Expose your local DB via ngrok (Recommended for testing)**
1. On your **local machine** (where PostgreSQL runs), open a terminal:
   ```
   ngrok tcp 5432
   ```
2. ngrok will give you a URL like: `tcp://0.tcp.ngrok.io:12345`
3. In this Kaggle notebook, set the DB environment variables **before** running the server cell:
   ```python
   os.environ['DB_HOST'] = '0.tcp.ngrok.io'   # from ngrok output
   os.environ['DB_PORT'] = '12345'             # from ngrok output
   os.environ['DB_NAME'] = 'pre_consultation_db'
   os.environ['DB_USER'] = 'postgres'
   os.environ['DB_PASSWORD'] = 'your_password'
   ```

**Option 2: Use a free cloud PostgreSQL**
- [Neon](https://neon.tech) — free tier, serverless PostgreSQL
- [Supabase](https://supabase.com) — free tier with PostgreSQL
- Set the DB_HOST/DB_PORT/DB_NAME/DB_USER/DB_PASSWORD env vars accordingly

**Option 3: Skip the DB (test conversation only)**
- The pipeline test cells (Steps 1-8) work without any database
- For frontend testing, you would need to modify `kiosk.py` to make DB calls optional (wrap in try/except)